In [ ]:
import pandas as pd
import numpy as np
import datetime

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.neighbors import KernelDensity
import plotly.express as px

import quantrix.modules as qx
from scipy.stats import linregress, norm, lognorm, beta, skew, kurtosis, contingency, mode

In [ ]:
data0 = pd.read_csv('CNYRUB_TOM_10m.csv', sep = ',')
data = data0[['open', 'close', 'volume', 'begin', 'end','high','low']]
data['begin'] = pd.to_datetime(data.begin)
data['end'] = pd.to_datetime(data.end)
data['begin_date'] = data.begin.dt.date
data['begin_time'] = data.begin.dt.time
data['end_date'] = data.end.dt.date
data['end_time'] = data.end.dt.time

total_volume = data.groupby(by='begin_date')['volume'].agg('sum')

data0 = data.merge(total_volume, on='begin_date')
data0['volume_%'] = data0.volume_x/data0.volume_y
data0['inventory_left'] = 1 - data0['volume_%']
data0['inventory_left_abs'] = data0.volume_y - data0.volume_x

for i in range(1,len(data0)):
    if data0['volume_y'].iloc[i-1] == data0['volume_y'].iloc[i]: #while we work in one day
        data0['inventory_left'].iloc[i] = data0['inventory_left'].iloc[i-1] - data0['volume_%'].iloc[i]
    else:
        pass

data0['vwap'] = [np.sum(data0['close'].iloc[:i+1]*data0['volume_x'].iloc[:i+1])/np.sum(data0['volume_x'].iloc[:i+1]) for i in range(len(data0))]
data0['av_price'] = 1/3*(data0.low+data0.high+data0.close)

In [ ]:
volumes = data0[['begin_date','begin_time','inventory_left']]
prices = data0[['begin_date', 'begin_time', 'open', 'close', 'high', 'low']]
list_2 = list(volumes.groupby(by='begin_time')['inventory_left'])
total_bytime2 = dict(list(data0.groupby(by='begin_time')['inventory_left']))

In [ ]:
av_price = data0.groupby(by='begin_time').agg('av_price').mean()
min_price = data0.groupby(by='begin_time').agg('close').min()
vwap = data0.groupby(by='begin_time').agg('vwap').mean()

pl = go.Figure()
pl = make_subplots(rows = 1, cols = 2)
pl.add_trace(go.Scatter(x = np.arange(0, 33), y = av_price, name = 'Average_price'), row = 1, col = 1)
pl.add_trace(go.Scatter(x = np.arange(0, 33), y = vwap, name = 'VWAP'), row = 1, col = 2)
pl.update_layout(
    title_text='Comparison of Realised Prices and VWAP',
    title_font=dict(size=18, color='#1a1a1a'),
    title_x=0.05,                    # Center the title
    title_xanchor='left',
    title_pad=dict(t=20, b=10),
    width=1500,
    height=700,
    margin=dict(l=80, r=50, t=90, b=70),
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=12, color='#333333'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.15,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        font=dict(size=11),
    ),
)
pl.update_xaxes(title_text='Time partitions', title_font=dict(size=12), row=1, col=1, showline=True, linecolor='black', linewidth=1, mirror=True)
pl.update_xaxes(title_text='Time partitions', title_font=dict(size=12), row=1, col=2, showline=True, linecolor='black', linewidth=1, mirror=True)
pl.update_yaxes(title_text='Price', title_font=dict(size=12), showline=True, linecolor='black', linewidth=1, mirror=True)


In [ ]:
S = list(prices.groupby(by='begin_time')[['open','close','high','low']])
s_t = np.array([np.mean(values[1].close) for values in S])
x_ = [values[0] for values in S]

In [ ]:
class strategy_():
    def __init__(self, sigma, x0, n_paths, xt = 0, T = 33, granularity = 1):
        self.sigma = sigma
        self.x0 = x0
        self.xt = xt
        self.T = T
        self.n_paths = n_paths
        self.granularity = granularity

    def simulation(self):
        pass

    def time_linspace(self):
        time_ = np.linspace(0,self.T,self.T*self.granularity+1)
        return time_

    def expectation(self):
        pass

    def vol(self):
        pass

    def naive_str(self):
        return np.array([self.x0/self.T]*self.T)

In [ ]:
class standart_bb(strategy_):
    def __init__(self, sigma, x0, n_paths, T=33):
        super().__init__(sigma, x0, n_paths, T)

    def simulation(self):
        w = np.array(qx.bm_simulations(self.n_paths, self.granularity, self.T))
        bridge = [self.sigma*(w[i] - self.time_linspace()/self.T*w[i][-1]) for i in range(self.n_paths)]
        X = self.x0*(1-self.time_linspace()/self.T) + bridge
        return X
    
    def expectation(self): # strategy value
        means_ = [np.mean(i) for i in self.simulation().T]
        return means_
    
    def vol(self):
        vola = [np.std(i) for i in self.simulation().T]
        return vola
    
    def simulation_adjusted(self):
        pass
    
class bb_with_bounds_simulation(standart_bb):
    def __init__(self, sigma, x0, n_paths, T=33):
        super().__init__(sigma, x0, n_paths, T)

    def simulation_adjusted(self):
        bb = self.simulation()

        X = []
        for i in range(self.n_paths):
            bi = bb[i]
            for j in range(len(bi)-1):
                if bi[j] <= 0:
                    #print('below zero at :', i, 'path ', j, 'point', len(bi)-j)
                    bi = list(bi)[:j] + [0]*(len(bi)-j)
                    pass
                elif bi[j] >= self.x0 and j > 0:
                    #print('above one at :', i, 'path ', j, 'point ', 'left points ', len(bi)-j)
                    bi = list(bi)[:j] + [2-point for point in bi[j:]]
                    pass
            X.append(bi)

        return np.array(X)
    
    def expectation(self): # strategy value
        means_ = [np.mean(i) for i in self.simulation_adjusted().T]
        return means_
    
    def vol(self):
        vola = [np.std(i) for i in self.simulation_adjusted().T]
        return vola
    
class bb_with_low_bound_strategy(standart_bb):
    def __init__(self, sigma, x0, n_paths, T=33):
        super().__init__(sigma, x0, n_paths, T)

    def simulation_adjusted(self):
        bb = self.simulation()

        X = []
        for i in range(self.n_paths):
            bi = bb[i]
            for j in range(len(bi)-1):
                if bi[j] <= 0:
                    #print('below zero at :', i, 'path ', j, 'point', len(bi)-j)
                    bi = list(bi)[:j] + [0]*(len(bi)-j)
                    pass
            X.append(bi)

        return np.array(X)
    
    def expectation(self): # strategy value
        means_ = [np.mean(i) for i in self.simulation_adjusted().T]
        return means_
    
    def vol(self):
        vola = [np.std(i) for i in self.simulation_adjusted().T]
        return vola

In [ ]:
class prices():
    def __init__(self, prices, h = 0.08):
        self.S = prices
        self.h = h
        self.x_ = np.linspace(0,1,len(self.S))

    def _gaus_kernel(self,x,xi):
        return np.exp(-0.5*(x-xi)**2/self.h**2)

    def _kd_regression(self, x):
        k_h = self._gaus_kernel(x, self.x_)
        numerator = np.dot(self.S, k_h)
        denominator = np.sum(k_h)
        return numerator/denominator

    def kde(self):
        y_estimated = [self._kd_regression(xi) for xi in self.x_]
        return np.array(y_estimated)

    def plot(self):
        gr = go.Figure()
        gr.add_trace(go.Scatter(x=self.x_, y=self.kde(), name='KDE'))
        gr.add_trace(go.Scatter(x=self.x_, y=self.S, name='real'))
        gr.update_layout(
            title='Approximation of historical prices behavior during a trading day.',
            title_font=dict(size=18, color='#1a1a1a'),
            title_x=0.05,
            title_xanchor='left',
            title_pad=dict(t=20, b=10),
            width=1500,
            height=700,
            margin=dict(l=80, r=50, t=90, b=70),
            paper_bgcolor='white',
            plot_bgcolor='white',
            font=dict(family='Arial, sans-serif', size=18, color='#333333'),
            legend=dict(
                orientation='h',
                yanchor='bottom',
                y=-0.15,
                xanchor='center',
                x=0.5,
                bgcolor='rgba(0,0,0,0)',
                font=dict(size=11),
            ),
        )
        gr.update_xaxes(title_text='Time partitions', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)
        gr.update_yaxes(title_text='Price', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)
        gr.show()


In [ ]:
class revenues():
    def __init__(self, predicted_quantity, price):
        self.predicted_quantity = predicted_quantity
        self.price = price
    
    def sold_(self):
        sold_ = np.array([abs(self.predicted_quantity[i]-self.predicted_quantity[i-1]) for i in range(1,len(self.predicted_quantity))])
        return sold_

In [ ]:
class calibration(strategy_):
    def __init__(self, sigma, x0, n_paths, T=33):
        super().__init__(sigma, x0, n_paths, T)
        self.sets = None

    def _split_data(self,data):
        training_data = data[:int(0.8*len(data))]
        test_set = data[int(0.8*len(data)): ]
        return training_data, test_set
    
    def _group_by_days(self,data):
        self.sets = self._split_data(data)
        training_data = self.sets[0].groupby(by='begin_time')['inventory_left']
        test_set = self.sets[1].groupby(by='begin_time')['inventory_left']
        return training_data, test_set
    
    def working_data_(self,data):
        working_data = self._group_by_days(data)[0]
        working_data = [1]+[np.mean(w[1]) for w in working_data]
        return working_data
    
    def plot(self,data):
        working_data = self.working_data_(data)
        plot = go.Figure()
        plot.add_trace(go.Scatter(x = self.time_linspace(), y = working_data))
        plot.update_layout(
            title='Calibration working data',
            title_font=dict(size=14, color='#1a1a1a'),
            title_x=0.0,
            title_xanchor='left',
            title_pad=dict(t=20, b=10),
            width=1500,
            height=700,
            margin=dict(l=80, r=50, t=90, b=70),
            paper_bgcolor='white',
            plot_bgcolor='white',
            font=dict(family='Arial, sans-serif', size=12, color='#333333'),
        )
        plot.update_xaxes(title_text='Time partitions', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)
        plot.update_yaxes(title_text='Inventory left', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)
        plot.show()

    def calibrate(self, data):
        return np.std(self.working_data_(data))


In [ ]:
sigma = calibration(sigma = 0, x0 = 1, n_paths = 1000).calibrate(data0)
test_sim2 = go.Figure()
# test_sim2 = make_subplots(rows=3,cols=1)

def objects(sigma, n_paths):
    obj = strategy_(sigma = sigma, x0 = 1, n_paths = n_paths)
    bb_st = standart_bb(sigma = sigma, x0 = 1, n_paths = n_paths)
    bb_with_bounds = bb_with_bounds_simulation(sigma = sigma, x0 = 1, n_paths = n_paths)
    bb_with_low_bound = bb_with_low_bound_strategy(sigma = sigma, x0 = 1, n_paths = n_paths)
    return obj, bb_st, bb_with_bounds, bb_with_low_bound

obj_ = objects(sigma, 1000)

obj = obj_[0]
bb_st = obj_[1]
bb_with_bounds = obj_[2]
bb_with_low_bound = obj_[3]

t_ = obj.time_linspace()

# for path in bb_st.simulation():
#         test_sim2.add_trace(go.Scatter(x = t_, y = path, line=dict(color='red', width=1), opacity=0.35, showlegend=False), row = 1, col = 1)

# for path in bb_with_bounds.simulation_adjusted():
#         test_sim2.add_trace(go.Scatter(x = t_, y = path, line=dict(color='green', width=1), opacity=0.35, showlegend=False), row = 2, col = 1)

for path in bb_with_low_bound.simulation_adjusted():
        test_sim2.add_trace(go.Scatter(x = t_, y = path, line=dict(color='blue', width=1), opacity=0.1, showlegend=False))

# test_sim2.add_trace(go.Scatter(x = t_, y = bb_st.expectation(), line=dict(color='black', width=2.5), mode='lines+markers', showlegend=False), row = 1, col = 1)
# test_sim2.add_trace(go.Scatter(x = t_, y = bb_with_bounds.expectation(), line=dict(color='black', width=2.5), mode='lines+markers', showlegend=False), row = 2, col = 1)
test_sim2.add_trace(go.Scatter(x = t_, y = bb_with_low_bound.expectation(), line=dict(color='black', width=2.5), mode='lines+markers', showlegend=False))
    
test_sim2.update_xaxes(title_text='Time partitions', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)
test_sim2.update_yaxes(title_text='Portfolio value', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)

test_sim2.update_layout(
    title_text='Simulated paths of the strategy with low bound and its expectation value',
    title_font=dict(size=18, color='#1a1a1a'),
    title_x=0.05,
    title_xanchor='left',
    title_pad=dict(t=20, b=10),
    width=1500,
    height=700,
    margin=dict(l=80, r=50, t=90, b=70),
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=14, color='#333333'),
 )

test_sim2.show()


In [ ]:
def upper_bound_prob(sigma, n_paths = 1000):
    bb_with_low_bound = bb_with_low_bound_strategy(sigma = sigma, x0 = 1, n_paths = n_paths)
    # Fraction of time steps above 1, averaged over paths
    fractions = [(len(np.where(path>1)[0])/(len(path))) for path in bb_with_low_bound.simulation_adjusted()]
    return np.mean(fractions)

up = go.Figure()
y_ = [upper_bound_prob(sigma/100, 1000) for sigma in range(1,101)]
x_ = [sigma/100 for sigma in range(1,101)]

up.add_trace(go.Scatter(x = x_, y = y_))
up.update_xaxes(title_text='Sigma values', title_font=dict(size=20), showline=True, linecolor='black', linewidth=1, mirror=True)
up.update_yaxes(title_text='P(X_t > 1)', title_font=dict(size=20), showline=True, linecolor='black', linewidth=1, mirror=True)
up.update_layout(
    title='The probability of exceeding initial portfolio value during the trading day for different sigma',
    title_font=dict(size=27, color='#1a1a1a'),
    title_x=0.05,
    title_xanchor='left',
    title_pad=dict(t=20, b=10),
    width=1500,
    height=700,
    margin=dict(l=80, r=50, t=90, b=70),
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=18, color='#333333'),
 )
up.show()


In [ ]:
def low_bound_prob(sigma, n_paths = 1000):
    bb_with_low_bound = bb_with_low_bound_strategy(sigma = sigma, x0 = 1, n_paths = n_paths)
    low = []
    tau = []
    paths = []
    for path in bb_with_low_bound.simulation_adjusted():
        filtered_path = path[path > 1]
        t_min = [i for i in range(len(path)) if path[i] >= 1][0]
        low.append(len(filtered_path)/len(path))
        tau.append(t_min)
        paths.append(path)
    return bb_with_low_bound.time_linspace(), np.mean(low), np.mean(tau) if tau else 0, paths

In [ ]:
adjust_for_time = bb_with_low_bound.simulation_adjusted().T
mins_cross = [np.min(w) for w in adjust_for_time]
for_revenue = [np.abs(mins_cross[i] - mins_cross[i-1]) for i in range(1,len(mins_cross))]
np.sum(for_revenue*prices(s_t).kde())

In [ ]:
def moments(el):  
    k1 = el
    setka = np.arange(0,1.1,0.1)
    moments = {}
    t = float('inf')
    m = float('inf')
    for tau in range(len(k1)):
        for i in setka[::-1]:
            if k1[tau] <= i:
                if k1[tau] <= m:
                    moments[tau] =  k1[tau]
                    t = tau
                    m = k1[tau]
            else:
                moments[tau] = m
    return moments

In [ ]:
k = bb_with_low_bound.simulation_adjusted()

mom = go.Figure()
revenue = []
means = []
for el in k:
    moment = moments(el)
    x_ = [xi for xi in moment.keys()]
    y_ = [yi for yi in moment.values()]
    means.append(y_)
    qty = np.array([np.abs(y_[i] - y_[i-1]) for i in range(1,len(y_))])
    p = np.array([prices(s_t).kde()[xi-1] for xi in x_[1:]])
    revenue.append((np.sum(p*qty),x_,y_))

r = max(revenue)
means = np.array(means).T
means = [np.mean(m) for m in means]
qty = np.array([np.abs(means[i] - means[i-1]) for i in range(1,len(means))])
p = np.array([prices(s_t).kde()[xi-1] for xi in x_[1:]])
print(p @ qty)

mom.add_trace(go.Scatter(x = r[1], y = r[2], mode='lines+markers', name = 'For max revenuue'))
mom.add_trace(go.Scatter(x = r[1], y = means, mode='lines+markers', name = 'Mean'))

mom.update_layout(
    title_text='Maximum revenue path and mean trajectory',
    title_font=dict(size=18, color='#1a1a1a'),
    title_x=0.05,
    title_xanchor='left',
    title_pad=dict(t=20, b=10),
    width=1500,
    height=1000,
    margin=dict(l=80, r=50, t=90, b=70),
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=18, color='#333333'),
 )
mom.update_xaxes(title_text='Time partitions', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)
mom.update_yaxes(title_text='Portfolio value', title_font=dict(size=14), showline=True, linecolor='black', linewidth=1, mirror=True)
mom.show()
print(r[0])


In [ ]:
mom = go.Figure()
i = 0
for r in revenue[:10]:
    i += 1
    mom.add_trace(go.Scatter(x = r[1], y = r[2], mode='lines+markers', name = f'path {i}'))
mom.update_layout(
    title_text='Monotonic strategy trajectories',
    title_font=dict(size=25, color='#1a1a1a'),
    title_x=0.05,
    title_xanchor='left',
    title_pad=dict(t=20, b=10),
    width=1500,
    height=1200,
    margin=dict(l=80, r=50, t=90, b=70),
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=20, color='#333333'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.1,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        font=dict(size=18),
    ),
    xaxis=dict(
        showgrid=True,
        gridwidth=0.01,
        gridcolor='rgba(0,0,0,0.01)',
    ),
    yaxis=dict(
        showgrid=True,
        gridwidth=0.5,
        gridcolor='rgba(0,0,0.5,0.5)',
        minor=dict(
            showgrid=True,
            gridwidth=0.5,
            gridcolor='rgba(0,0,0.5,0.5)',
            ticklen=5,
        )
    )
 )
mom.update_xaxes(title_text='Time partitions', title_font=dict(size=20), showline=True, linecolor='black', linewidth=1, mirror=True)
mom.update_yaxes(title_text='Portfolio value', title_font=dict(size=20), showline=True, linecolor='black', linewidth=1, mirror=True)
mom.show()
